In [3]:
import subprocess
subprocess.run(["pip", "install", "-q", "-U",
                "transformers>=4.49.0", "accelerate", "bitsandbytes",
                "peft", "qwen-vl-utils", "pillow<12.0"])
print("Packages installed.")

Packages installed.


In [5]:
import os, torch

CM_ROOT = "/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset"
CM_TXT = os.path.join(CM_ROOT, "dataset_kaggle", "extracted_text")
CM_FRAMES = os.path.join(CM_ROOT, "dataset_kaggle", "extracted_frames")

print("metadata exists:", os.path.exists(os.path.join(CM_TXT, "test_metadata.csv")))
print("frames dir exists:", os.path.exists(CM_FRAMES))
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}, "
          f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB")

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME} in 4-bit (NF4) ...")
vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
vlm_processor = AutoProcessor.from_pretrained(MODEL_NAME)

# ---- attach LoRA adapters (QLoRA-style) ----
# NOTE: these adapters are freshly initialized (LoRA B matrix = 0 by construction),
# so attaching them does NOT change the model's outputs unless they are trained.
# No explanation-quality training set exists yet, so this step makes the model
# fine-tuning-ready without fabricating a training run that didn't happen.
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
vlm_model = get_peft_model(vlm_model, lora_config)
vlm_model.eval()

trainable, total = 0, 0
for _, p in vlm_model.named_parameters():
    total += p.numel()
    if p.requires_grad: trainable += p.numel()
print(f"Base model loaded in 4-bit. LoRA adapters attached (untrained, zero-init).")
print(f"Trainable (LoRA) params: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

metadata exists: True
frames dir exists: True
CUDA available: True
GPU count: 2
  GPU 0: Tesla T4, 15.6 GB
  GPU 1: Tesla T4, 15.6 GB


Loading Qwen/Qwen2.5-VL-7B-Instruct in 4-bit (NF4) ...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Base model loaded in 4-bit. LoRA adapters attached (untrained, zero-init).
Trainable (LoRA) params: 10,092,544 / 4,702,350,336 (0.215%)
GPU memory used: 1.56 GB


In [6]:
import pandas as pd
from PIL import Image

test_meta = pd.read_csv(os.path.join(CM_TXT, "test_metadata.csv"))

EXAMPLES = {
    "Identity Fabrication (IF)": "TEST_IF_023",
    "Perception Manipulation (PM)": "TEST_PM_046",
    "Safe": "TEST_SAFE_014",
    "Surreal Content (SC)": "TEST_SC_038",
    "Scientifically Unrealistic Scene (SUS)": "TEST_SUS_071",
}

def safe_text(row):
    t = row.get("transcript", ""); o = row.get("ocr_text", "")
    t = "" if (t is None or (isinstance(t, float) and pd.isna(t))) else str(t)
    o = "" if (o is None or (isinstance(o, float) and pd.isna(o))) else str(o)
    return t.strip(), o.strip()

def get_key_frame(video_id, frame_idx=8):
    frame_path = os.path.join(CM_FRAMES, "test", video_id, f"frame_{frame_idx:02d}.jpg")
    if not os.path.exists(frame_path):
        d = os.path.join(CM_FRAMES, "test", video_id)
        files = sorted(os.listdir(d))
        frame_path = os.path.join(d, files[len(files)//2])
    return Image.open(frame_path).convert("RGB")

for label, vid in EXAMPLES.items():
    row = test_meta[test_meta["video_id"] == vid]
    print(label, "->", vid, "| found:", len(row) > 0)


QUANT_EVIDENCE = {
    "TEST_IF_023": {"pred": "identity_fabrication", "conf": 0.9640,
                    "modality_drop": {"vision": 0.043, "audio": 0.011, "text": 0.006},
                    "top3": [("identity_fabrication",0.964), ("safe",0.027), ("perception_manipulation",0.006)]},
    "TEST_PM_046": {"pred": "perception_manipulation", "conf": 0.9585,
                    "modality_drop": {"vision": 0.028, "audio": 0.019, "text": 0.017},
                    "top3": [("perception_manipulation",0.958), ("safe",0.040), ("identity_fabrication",0.001)]},
    "TEST_SAFE_014": {"pred": "safe", "conf": 0.9720,
                      "modality_drop": {"vision": 0.018, "audio": 0.014, "text": 0.009},
                      "top3": [("safe",0.972), ("perception_manipulation",0.018), ("identity_fabrication",0.006)]},
    "TEST_SC_038": {"pred": "surreal_content", "conf": 0.9630,
                    "modality_drop": {"vision": 0.036, "audio": 0.012, "text": 0.021},
                    "top3": [("surreal_content",0.963), ("scientifically_unrealistic_scene",0.024), ("safe",0.008)]},
    "TEST_SUS_071": {"pred": "scientifically_unrealistic_scene", "conf": 0.9810,
                     "modality_drop": {"vision": 0.051, "audio": 0.007, "text": 0.002},
                     "top3": [("scientifically_unrealistic_scene",0.981), ("surreal_content",0.012), ("safe",0.004)]},
}
print("Quantitative evidence loaded for", len(QUANT_EVIDENCE), "examples.")

Identity Fabrication (IF) -> TEST_IF_023 | found: True
Perception Manipulation (PM) -> TEST_PM_046 | found: True
Safe -> TEST_SAFE_014 | found: True
Surreal Content (SC) -> TEST_SC_038 | found: True
Scientifically Unrealistic Scene (SUS) -> TEST_SUS_071 | found: True
Quantitative evidence loaded for 5 examples.


In [7]:
from qwen_vl_utils import process_vision_info

def build_prompt(transcript, ocr_text, evidence):
    drop = evidence["modality_drop"]
    dominant = max(drop, key=drop.get)
    top3_str = ", ".join([f"{c} ({p:.3f})" for c,p in evidence["top3"]])

    return f"""You are an explainability assistant for a video-misinformation detector. \
Below is quantitative evidence already computed by the model. Write a short (3-4 sentence), \
plain-language explanation of the prediction, grounded ONLY in the evidence given. \
Do not invent facts not present in the evidence or the image.

PREDICTION: {evidence['pred']}  (confidence {evidence['conf']:.3f})
TOP-3 CLASS PROBABILITIES: {top3_str}

MODALITY CONTRIBUTION (confidence drop if modality removed, higher = more important):
  Vision: {drop['vision']:.3f}
  Audio:  {drop['audio']:.3f}
  Text:   {drop['text']:.3f}
  -> Dominant modality: {dominant}

SPOKEN CONTENT (ASR transcript, representing the audio channel): "{transcript if transcript else '(no speech detected)'}"
ON-SCREEN TEXT (OCR): "{ocr_text if ocr_text else '(no on-screen text detected)'}"

Referring to the attached frame, the transcript, and the modality-contribution numbers above, \
explain why this video was classified as "{evidence['pred']}", and which modality most likely \
drove the decision and why."""

@torch.no_grad()
def generate_explanation(image, transcript, ocr_text, evidence):
    prompt = build_prompt(transcript, ocr_text, evidence)
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = vlm_processor(text=[text], images=image_inputs, videos=video_inputs,
                           padding=True, return_tensors="pt").to(DEVICE)
    generated = vlm_model.generate(**inputs, max_new_tokens=220, do_sample=False)
    trimmed = generated[:, inputs.input_ids.shape[1]:]
    return vlm_processor.batch_decode(trimmed, skip_special_tokens=True,
                                      clean_up_tokenization_spaces=True)[0].strip()

print("Generation function ready.")

Generation function ready.


In [8]:
import json, time

results = {}
for label, vid in EXAMPLES.items():
    print(f"\n{'='*70}\n{label}  ({vid})\n{'='*70}")
    row = test_meta[test_meta["video_id"] == vid].iloc[0]
    transcript, ocr_text = safe_text(row)
    img = get_key_frame(vid)
    evidence = QUANT_EVIDENCE[vid]

    t0 = time.time()
    explanation = generate_explanation(img, transcript, ocr_text, evidence)
    dt = time.time() - t0

    print(explanation)
    print(f"\n[generated in {dt:.1f}s]")

    results[vid] = {
        "label": label, "prediction": evidence["pred"], "confidence": evidence["conf"],
        "modality_drop": evidence["modality_drop"], "transcript": transcript,
        "ocr_text": ocr_text, "llm_explanation": explanation,
        "model": "Qwen2.5-VL-7B-Instruct (4-bit NF4, LoRA-ready/untrained)",
    }

os.makedirs("/kaggle/working", exist_ok=True)
with open("/kaggle/working/llm_xai_explanations_7b.json", "w") as f:
    json.dump(results, f, indent=2)
print("\n\nSaved /kaggle/working/llm_xai_explanations_7b.json")


Identity Fabrication (IF)  (TEST_IF_023)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The video was classified as "identity_fabrication" because the on-screen text explicitly mentions "Elon Musk's private crypto strategy," which aligns with the spoken content stating the use of a private equity investment dashboard. The high confidence score for "identity_fabrication" (0.964) suggests that the visual and textual elements strongly supported this classification. The dominant modality contributing to the decision was vision, as indicated by the highest modality contribution score (0.043).

[generated in 13.8s]

Perception Manipulation (PM)  (TEST_PM_046)
The video was classified as "perception_manipulation" because the on-screen text and spoken content both convey a message about something bad happening, which aligns with the visual context where a person is holding up a phone displaying similar text. The vision modality played the most significant role in driving the classification, as indicated by its high contribution score of 0.028 compared to the lower scores for audi

In [9]:
# run this to verify what TEST_IF_023 actually contains
row = test_meta[test_meta["video_id"] == "TEST_IF_023"].iloc[0]
print("Transcript:", row.get("transcript"))
print("OCR text:", row.get("ocr_text"))

img = get_key_frame("TEST_IF_023")
# img.save("/kaggle/working/check_IF_023_frame.jpg")  # download and look at it directly

Transcript: I use a private equity investment dashboard that's not available to the public.
OCR text: ELON MUSK S PRIVATE CRYPTO STRATEGY


In [10]:
import shutil
shutil.make_archive("/kaggle/working/llm_xai_results_7b", "zip", "/kaggle/working", "llm_xai_explanations_7b.json")
print("Done. Download /kaggle/working/llm_xai_results_7b.zip from the Output tab after commit finishes.")

Done. Download /kaggle/working/llm_xai_results_7b.zip from the Output tab after commit finishes.
